# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [40]:
# imports
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [41]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [42]:
# set up environment
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [57]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = "You are an assistant that analyzes what a provided code samples does. \
Please explain what this code does and why? provides a short summary, ignoring text that might be navigation related. \
try as much as possible not to exceed 1000 words. \
Respond in markdown."

In [58]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [59]:
# A function that writes a User Prompt that asks for summaries of websites:

def user_prompt_for(question):
    user_prompt = f"You are looking at a code sample provided"
    # user_prompt += "\n"
    user_prompt += f"{question}"
    return user_prompt

In [60]:
print(user_prompt_for(question))

You are looking at a code sample provided
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}



In [61]:
# Format message content

def messages_for(question):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(question)}
    ]

In [62]:
# Get gpt-4o-mini to answer, with streaming
def stream_code_explain(question):
    stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages_for(question),
        # below code is use to make typewriter feature enable when returning the response
        stream=True)
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [63]:
stream_code_explain(question)

The provided code snippet utilizes Python's `yield from` statement in conjunction with a set comprehension. Here’s a breakdown of what this code does and why it is structured in this way:

### Code Explanation

1. **Set Comprehension**:
    python
    {book.get("author") for book in books if book.get("author")}
    
   - This portion of the code creates a set of authors from a collection named `books`.
   - It iterates over each `book` in `books` (which is assumed to be a list or iterable of dictionaries).
   - For each `book`, it retrieves the value associated with the key `"author"` using `book.get("author")`.
   - The `if book.get("author")` condition filters out any `book` entries that do not have an author (i.e., `None` or empty strings) to avoid adding invalid or missing authors.

2. **`yield from`**:
    python
    yield from <set_comprehension>
    
   - The `yield from` expression is used in Python's generators. It yields each element from the iterable (in this case, the set created by the set comprehension) to the generator that calls this expression.
   - This means that instead of returning the whole set, it returns each author one at a time as they are requested.

### Summary

In summary, the code creates a generator function that yields unique authors from a list of book dictionaries. It filters out any entries that do not have an author and avoids adding duplicates by using a set. This structure is efficient for iterating over authors since it combines filtering, deduplication, and yielding elements in a single line.

In [83]:
# Get Llama 3.2 to answer
import ollama


def ollama_code_explain(question):
    stream = ollama.chat(model=MODEL_LLAMA, messages=messages_for(question),
        # below code is use to make typewriter feature enable when returning the response
        # then this available for chatgpt llm only
        # stream=True
                        )
    display_handle = display(Markdown(""), display_id=True)
    update_display(Markdown(stream['message']['content']), display_id=display_handle.display_id)
    # return stream['message']['content']

In [84]:
ollama_code_explain(question)

**Summary**
---------------

This code snippet uses a combination of dictionary methods (`get`) and generator expressions to extract a list of authors from a collection of books.

### Code Breakdown
-----------------

Let's analyze the code:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

*   `yield from`: This is a Python keyword that allows us to delegate iteration over an iterator or iterable to another iterator. In this case, it's used to yield values from the dictionary comprehension.
*   `{...}`: This is a dictionary literal, which creates a new dictionary with key-value pairs.
*   `for book in books`: This loop iterates over each book in the collection (`books`).
*   `if book.get("author")`: This condition filters out any book that doesn't have an "author" key. The `.get()` method returns the value associated with a given key, or `None` if the key is not present.
*   `book.get("author")`: Inside the dictionary comprehension, this extracts the author's name from each book and uses it as the value in the resulting dictionary.

### Explanation
-----------------

The code is essentially saying: "Take the authors' names from the books, and yield them one by one." However, instead of using a traditional `for` loop to iterate over the authors, we're using a generator expression inside a dictionary comprehension. This allows us to lazily compute the list of authors without loading all the data into memory at once.

**Why?**
----------

Using `yield from` with a generator expression (the part between `for book in books`) is more memory-efficient than creating a large list of authors and then yielding them one by one. It's especially useful when dealing with large datasets, as it allows us to process the data in smaller chunks.

**Example Use Case**
--------------------

Suppose we have a collection of books and want to extract their authors:
```python
books = [
    {"title": "Book 1", "author": "John Doe"},
    {"title": "Book 2", "author": None},
    {"title": "Book 3", "author": "Jane Smith"}
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

This will output:
```
John Doe
Jane Smith
```